# 01 Mental Models

Framework vs runtime vs harness, when to use LangChain, LangGraph, or Deep Agents.


In [6]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "shared").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# Pick up edits to shared/ without a full kernel restart (re-run this cell)
for _name in (
    "shared.dataflow",
    "shared.notebook_display",
    "shared.llm",
    "shared.bootcamp_fixtures",
):
    importlib.reload(importlib.import_module(_name))

print(f"Project root: {ROOT}")


Project root: c:\Users\Azooo\langchain-bootcamp


## Learning objectives

- Explain Agent = Model + Harness
- Choose the right layer for a task
- Map bootcamp travel-assistant scenario across all three layers


In [7]:
import os

def env_status():
    keys = {
        "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
        "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
        "LLM_PROVIDER": os.getenv("LLM_PROVIDER", "deepseek"),
        "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
        "LANGSMITH_API_KEY": bool(os.getenv("LANGSMITH_API_KEY")),
    }
    for k, v in keys.items():
        print(f"{k}: {v}")
    return keys

ENV = env_status()


ZAI_API_KEY: True
OPENAI_API_KEY: False
ANTHROPIC_API_KEY: False
TAVILY_API_KEY: False
LANGSMITH_API_KEY: True
LLM_PROVIDER: deepseek


In [ ]:
import os
from shared.llm import get_model

def require_llm():
    return get_model()


### Visual

```
Deep Agents to planning, filesystem, subagents, compression
LangChain to create_agent() to CompiledStateGraph
LangGraph to StateGraph when graph shape is not one agent loop
```


## Pick the right layer

Same travel problem, different abstraction levels. create_agent() is LangGraph under the hood; raw StateGraph is for custom node graphs.


Run the demo below to verify the behavior.


In [9]:
TASKS = [
    ("Standard tool loop", "create_agent()"),
    ("Extra fields on agent loop", "create_agent + middleware / state_schema"),
    ("Custom pipeline / Send / no messages channel", "StateGraph"),
    ("Long task + files + subagents", "Deep Agents"),
    ("Pure RAG, no agent", "LangChain retriever + chain"),
]
for need, layer in TASKS:
    print(f"{need:40} to {layer}")


Standard tool loop                       to create_agent()
Extra fields on agent loop               to create_agent + middleware / state_schema
Custom pipeline / Send / no messages channel to StateGraph
Long task + files + subagents            to Deep Agents
Pure RAG, no agent                       to LangChain retriever + chain


**Reflect:** Use raw StateGraph when the graph shape is not model and tools, not because create_agent lacks state.


## Same feature, three APIs

Memory, HITL, and streaming exist at every layer, different entry points.


Run the demo below to verify the behavior.


In [10]:
FEATURES = {
    "memory": ("checkpointer=", "compile(checkpointer=)", "checkpointer= + backends"),
    "HITL": ("HumanInTheLoopMiddleware", "interrupt()", "interrupt_on="),
    "streaming": (".stream()", "stream_events", "same + subagent streams"),
}
for feat, (lc, lg, da) in FEATURES.items():
    print(f"{feat}: LangChain={lc} | LangGraph={lg} | DeepAgents={da}")


memory: LangChain=checkpointer= | LangGraph=compile(checkpointer=) | DeepAgents=checkpointer= + backends
HITL: LangChain=HumanInTheLoopMiddleware | LangGraph=interrupt() | DeepAgents=interrupt_on=
streaming: LangChain=.stream() | LangGraph=stream_events | DeepAgents=same + subagent streams
